In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2005_2006"
    / "Labs"
)

xpt_file = DATA_DIR / "TRIGLY_D.xpt"
csv_file = DATA_DIR / "TRIGLY_D_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Untouched source for final fidelity comparison
source_df = df.copy()


print("=" * 80)
print("NHANES 2005-2006 TRIGLY_D XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. STRUCTURE CHECK
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "WTSAF2YR",
    "LBXTR",
    "LBDTRSI",
    "LBDLDL",
    "LBDLDLSI",
    "LBXAPB",
    "LBDAPBSI"
]


if len(df) != 3352:
    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) != 8:
    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "TRIGLY_D columns/order differ.\n"
        f"Actual: {df.columns.tolist()}"
    )


if df.columns.duplicated().any():
    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: TRIGLY_D structure = "
    "3,352 rows x 8 columns."
)


# ============================================================
# 4. DEFINE LAB / WEIGHT MEASUREMENT FIELDS
# ============================================================

#
# CRITICAL:
#
# ALL of these are measurements/weights.
#
# DO NOT:
# - convert to Int64
# - round()
# - recalculate
# - trim .0
# - apply float_format
#

MEASUREMENT_COLUMNS = [
    "WTSAF2YR",
    "LBXTR",
    "LBDTRSI",
    "LBDLDL",
    "LBDLDLSI",
    "LBXAPB",
    "LBDAPBSI"
]


# ============================================================
# 5. CHECK KNOWN XPORT TINY-VALUE ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "True numeric zeros:",
    f"{int(true_zero_counts.sum()):,}"
)

print(
    "Tiny artifact occurrences:",
    f"{int(tiny_counts.sum()):,}"
)


print("\nTiny values by variable:")

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. EXPECTED TINY COUNT
# ============================================================

EXPECTED_TINY_COUNTS = {
    "WTSAF2YR": 482
}


ZERO_ALLOWED_COLUMNS = [
    "WTSAF2YR"
]


# ============================================================
# 7. SAFETY CHECK
# ============================================================

actual_tiny_columns = set(
    tiny_counts[
        tiny_counts > 0
    ].index
)


expected_tiny_columns = set(
    ZERO_ALLOWED_COLUMNS
)


unexpected = (
    actual_tiny_columns
    -
    expected_tiny_columns
)


if unexpected:

    raise ValueError(
        "STOP: Tiny values found in unexpected "
        "TRIGLY_D variables:\n"
        + str(sorted(unexpected))
    )


missing_expected = (
    expected_tiny_columns
    -
    actual_tiny_columns
)


if missing_expected:

    raise ValueError(
        "STOP: Expected tiny-value variables missing:\n"
        + str(sorted(missing_expected))
    )


if int(
    tiny_counts["WTSAF2YR"]
) != 482:

    raise ValueError(
        "WTSAF2YR tiny count differs from expected 482."
    )


if int(
    tiny_counts.sum()
) != 482:

    raise ValueError(
        "Unexpected total tiny-value count."
    )


print(
    "\nPASS: All 482 tiny values occur only "
    "in WTSAF2YR."
)


# ============================================================
# 8. CONFIRM LAB RESULTS HAVE NO TINY ARTIFACT
# ============================================================

LAB_COLUMNS = [
    "LBXTR",
    "LBDTRSI",
    "LBDLDL",
    "LBDLDLSI",
    "LBXAPB",
    "LBDAPBSI"
]


lab_tiny_total = int(
    df[LAB_COLUMNS]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


if lab_tiny_total != 0:

    raise ValueError(
        "STOP: Tiny artifact found in "
        "triglyceride/LDL/ApoB result fields."
    )


print(
    "PASS: No tiny artifact in laboratory result fields."
)


# ============================================================
# 9. RESTORE VALIDATED WTSAF2YR ZEROS
# ============================================================

mask = (
    df["WTSAF2YR"]
    .eq(TINY_VALUE)
)


print(
    "\nWTSAF2YR tiny values -> 0:",
    int(mask.sum())
)


df.loc[
    mask,
    "WTSAF2YR"
] = 0.0


# ============================================================
# 10. VERIFY RESTORED ZEROS
# ============================================================

wtsaf_zero_count = int(
    df["WTSAF2YR"]
    .eq(0)
    .sum()
)


print(
    "WTSAF2YR zeros after restoration:",
    wtsaf_zero_count
)


if wtsaf_zero_count != 482:

    raise ValueError(
        "WTSAF2YR zero restoration failed."
    )


print(
    "PASS: 482 WTSAF2YR zeros restored."
)


# ============================================================
# 11. CONFIRM NO TINY ARTIFACT REMAINS
# ============================================================

remaining_tiny = int(
    df.select_dtypes(
        include=[np.number]
    )
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nRemaining tiny artifacts:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny-value artifact remains."
    )


print(
    "PASS: No tiny-value artifact remains."
)


# ============================================================
# 12. SEQN VALIDATION
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


print("\n--- SEQN ---")

print(
    "Missing:",
    int(df["SEQN"].isna().sum())
)

print(
    "Duplicates:",
    int(df["SEQN"].duplicated().sum())
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)


if (
    int(df["SEQN"].isna().sum()) != 0
    or
    int(df["SEQN"].duplicated().sum()) != 0
    or
    df["SEQN"].min() != 31130
    or
    df["SEQN"].max() != 41474
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 13. CDC RANGE / COMPLETENESS CHECKS
# ============================================================

LAB_CHECKS = {

    "WTSAF2YR": {
        "valid": 3352,
        "missing": 0,
        "min": 0.0,
        "max": 327394.027
    },

    "LBXTR": {
        "valid": 3089,
        "missing": 263,
        "min": 19.0,
        "max": 1600.0
    },

    "LBDTRSI": {
        "valid": 3089,
        "missing": 263,
        "min": 0.215,
        "max": 18.064
    },

    "LBDLDL": {
        "valid": 3026,
        "missing": 326,
        "min": 19.0,
        "max": 328.0
    },

    "LBDLDLSI": {
        "valid": 3026,
        "missing": 326,
        "min": 0.491,
        "max": 8.482
    },

    "LBXAPB": {
        "valid": 3106,
        "missing": 246,
        "min": 24.0,
        "max": 345.0
    },

    "LBDAPBSI": {
        "valid": 3106,
        "missing": 246,
        "min": 0.24,
        "max": 3.45
    }
}


print("\n--- CDC LAB CHECKS ---")


for col, spec in LAB_CHECKS.items():

    s = df[col]


    if int(
        s.notna().sum()
    ) != spec["valid"]:

        raise ValueError(
            f"{col}: valid count differs."
        )


    if int(
        s.isna().sum()
    ) != spec["missing"]:

        raise ValueError(
            f"{col}: missing count differs."
        )


    if not np.isclose(
        s.min(),
        spec["min"],
        rtol=0,
        atol=0
    ):

        raise ValueError(
            f"{col}: minimum differs."
        )


    if not np.isclose(
        s.max(),
        spec["max"],
        rtol=0,
        atol=0
    ):

        raise ValueError(
            f"{col}: maximum differs."
        )


    print(
        f"PASS: {col:<10}"
        f" range={s.min()}-{s.max()}"
        f" valid={int(s.notna().sum()):,}"
        f" missing={int(s.isna().sum()):,}"
    )


# ============================================================
# 14. REPORT LAB DECIMAL BEHAVIOR
# ============================================================

print("\n--- LAB DECIMAL / .0 REPORT ---")


for col in MEASUREMENT_COLUMNS:

    values = (
        df[col]
        .dropna()
    )


    fractional = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    dot_zero_count = sum(
        str(float(value)).endswith(".0")
        for value in values
    )


    print(
        f"{col:<10}"
        f" valid={len(values):,}"
        f" fractional={int(fractional.sum()):,}"
        f" .0={dot_zero_count:,}"
    )


#
# Expected pattern after WTSAF2YR correction:
#
# WTSAF2YR   .0 = 487
# LBXTR      .0 = 3,089
# LBDTRSI    .0 = 0
# LBDLDL     .0 = 3,026
# LBDLDLSI   .0 = 21
# LBXAPB     .0 = 3,106
# LBDAPBSI   .0 = 42
#


# ============================================================
# 15. IMPORTANT: DO NOT RECOMPUTE DERIVED VARIABLES
# ============================================================

#
# CDC documents:
#
# LBDTRSI = LBXTR * 0.01129
#
# LBDLDLSI = LBDLDL * 0.02586
#
# LBDLDL itself is calculated using:
#
# Total cholesterol
# - HDL cholesterol
# - triglycerides / 5
#
# BUT:
#
# We do NOT recalculate any of these.
#
# Released XPT values remain source of truth.
#

print(
    "\nPASS: Derived laboratory variables "
    "will not be recalculated."
)


# ============================================================
# 16. BUILD EXPECTED FIDELITY SOURCE
# ============================================================

#
# Expected source =
#
# original XPT
# +
# ONLY WTSAF2YR tiny -> 0.0
#

expected_df = source_df.copy()


expected_df.loc[
    expected_df["WTSAF2YR"]
    .eq(TINY_VALUE),
    "WTSAF2YR"
] = 0.0


# ============================================================
# 17. CLEAN ONLY SEQN
# ============================================================

#
# SEQN:
#
# 31130.0 -> 31130
#
# EVERYTHING ELSE remains a float measurement.
#

df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print(
    "\nPASS: SEQN converted to integer formatting."
)

print(
    "PASS: All lab/weight fields remain floats."
)


# ============================================================
# 18. VALUE FIDELITY BEFORE EXPORT
# ============================================================

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


value_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    if col == "SEQN":

        current_values = (
            df[col]
            .astype("Float64")
            .to_numpy(
                dtype=float,
                na_value=np.nan
            )
        )

    else:

        current_values = (
            df[col]
            .to_numpy()
        )


    if not np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        value_failures.append(
            col
        )


if value_failures:

    raise ValueError(
        "Unexpected value changes in:\n"
        + str(value_failures)
    )


print(
    "PASS: All 8 variables preserved exactly "
    "after WTSAF2YR zero restoration."
)


# ============================================================
# 19. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO float_format
# NO round()
# NO recalculation
# NO decimal trimming
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 20. RE-READ WITH ROUND-TRIP FLOAT PARSING
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 21. XPT -> CSV NUMERIC FIDELITY
# ============================================================

print("\n--- XPT -> CSV NUMERIC FIDELITY ---")


numeric_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )

    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    if not np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        numeric_failures.append(
            col
        )


if numeric_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found:\n"
        + str(numeric_failures)
    )


print(
    "PASS: All 8 CSV columns numerically "
    "match expected XPT values."
)


# ============================================================
# 22. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )

    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\nMissing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 23. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(f)
    raw_rows = list(reader)


# ============================================================
# 24. SEQN RAW FORMAT CHECK
# ============================================================

bad_seqn_dot_zero = 0
bad_seqn_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    value = row["SEQN"]


    if (
        value != ""
        and
        value.endswith(".0")
    ):

        bad_seqn_dot_zero += 1


        if len(
            bad_seqn_examples
        ) < 20:

            bad_seqn_examples.append(
                (
                    row_number,
                    value
                )
            )


print(
    "\nSEQN values ending '.0':",
    bad_seqn_dot_zero
)


if bad_seqn_dot_zero != 0:

    print(
        "Examples:",
        bad_seqn_examples
    )

    raise ValueError(
        "SEQN contains unnecessary .0 formatting."
    )


print(
    "PASS: SEQN has clean integer formatting."
)


# ============================================================
# 25. STRICT RAW LAB / WEIGHT FIDELITY
# ============================================================

#
# Compare actual CSV token against:
#
# str(float(expected XPT value))
#
# Therefore:
#
# LBXTR:
# 100.0 -> 100.0
#
# LBDTRSI:
# 1.129 -> 1.129
#
# LBDLDL:
# 95.0 -> 95.0
#
# LBDLDLSI:
# 2.46 -> 2.46
#
# LBXAPB:
# 90.0 -> 90.0
#
# WTSAF2YR:
# tiny -> 0.0
#
# No beautification is permitted.
#

measurement_mismatches = {
    col: 0
    for col in MEASUREMENT_COLUMNS
}


mismatch_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in MEASUREMENT_COLUMNS:

        source_value = (
            expected_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(source_value)
            else str(
                float(source_value)
            )
        )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            measurement_mismatches[
                col
            ] += 1


            if len(
                mismatch_examples
            ) < 20:

                mismatch_examples.append(
                    (
                        i + 2,
                        col,
                        expected_text,
                        actual_text
                    )
                )


bad_measurements = {
    col: count
    for col, count
    in measurement_mismatches.items()
    if count > 0
}


print(
    "Lab/weight raw-text mismatches:",
    bad_measurements
)


if bad_measurements:

    print(
        "Examples:",
        mismatch_examples
    )

    raise ValueError(
        "Laboratory/weight formatting changed."
    )


print(
    "PASS: All laboratory and weight values "
    "appear AS IS in CSV."
)


# ============================================================
# 26. LEGITIMATE .0 REPORT
# ============================================================

dot_zero_counts = {
    col: 0
    for col in MEASUREMENT_COLUMNS
}


for row in raw_rows:

    for col in MEASUREMENT_COLUMNS:

        value = row[col]


        if (
            value != ""
            and
            value.endswith(".0")
        ):

            dot_zero_counts[
                col
            ] += 1


print(
    "\nLegitimate measurement values ending '.0':"
)


for col, count in dot_zero_counts.items():

    print(
        f"{col:<10}: {count:,}"
    )


EXPECTED_DOT_ZERO_COUNTS = {

    "WTSAF2YR": 487,

    "LBXTR": 3089,

    "LBDTRSI": 0,

    "LBDLDL": 3026,

    "LBDLDLSI": 21,

    "LBXAPB": 3106,

    "LBDAPBSI": 42
}


if dot_zero_counts != EXPECTED_DOT_ZERO_COUNTS:

    raise ValueError(
        "Unexpected lab .0 representation counts."
    )


print(
    "PASS: Legitimate measurement .0 values "
    "preserved exactly."
)


# ============================================================
# 27. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = (
            value.lower()
        )


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        "Examples:",
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 28. FINAL TINY TEXT CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


if "5.397605346934028e-79" in csv_text:

    raise ValueError(
        "Tiny XPORT artifact remains in CSV."
    )


print(
    "PASS: Tiny artifact absent from CSV."
)


# ============================================================
# 29. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("TRIGLY_D CONVERSION COMPLETE")
print("=" * 80)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "Validated tiny -> zero:",
    482
)

print(
    "Tiny-value variable:",
    "WTSAF2YR only"
)

print(
    "SEQN:",
    "integer formatting"
)

print(
    "Lab/weight fields:",
    "preserved AS IS"
)

print(
    "XPT -> CSV numeric differences:",
    "0 expected"
)

print(
    "Missing-position differences:",
    "0 expected"
)

print(
    "Raw lab/weight differences:",
    "0 expected"
)

print(
    "SEQN .0:",
    "0 expected"
)

NHANES 2005-2006 TRIGLY_D XPT -> CSV
Rows: 3,352
Columns: 8
PASS: TRIGLY_D structure = 3,352 rows x 8 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 482

Tiny values by variable:
WTSAF2YR    482

PASS: All 482 tiny values occur only in WTSAF2YR.
PASS: No tiny artifact in laboratory result fields.

WTSAF2YR tiny values -> 0: 482
WTSAF2YR zeros after restoration: 482
PASS: 482 WTSAF2YR zeros restored.

Remaining tiny artifacts: 0
PASS: No tiny-value artifact remains.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 31130.0
Maximum: 41474.0
PASS: SEQN validated.

--- CDC LAB CHECKS ---
PASS: WTSAF2YR   range=0.0-327394.027 valid=3,352 missing=0
PASS: LBXTR      range=19.0-1600.0 valid=3,089 missing=263
PASS: LBDTRSI    range=0.215-18.064 valid=3,089 missing=263
PASS: LBDLDL     range=19.0-328.0 valid=3,026 missing=326
PASS: LBDLDLSI   range=0.491-8.482 valid=3,026 missing=326
PASS: LBXAPB     range=24.0-345.0 valid=3,106 missing=246
PASS: LBDAPBS